In [ ]:
!pip install git+https://github.com/the-aerospace-corporation/radiomana.git

import os
import math
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import radiomana

folder_path = "/anvil/projects/x-cis220051/corporate/aerospace-rf/fiot_highway2-main"
os.environ["DSET_FIOT_HIGHWAY2"] = folder_path

In [ ]:
import radiomana 
from collections import Counter
dataset = radiomana.Highway2Dataset()

CLASS_LABELS = {
    0: "None (Type 1)",
    1: "None (Type 2)",
    2: "None (Type 3)",
    3: "None (Type 4)",
    4: "Chirp, high distance",
    5: "Chirp, medium distance",
    6: "Chirp, small distance",
    7: "Cigarette lighter 1",
    8: "Cigarette lighter 2",
}
EXPECTED_SHAPE = (512, 243)
RNG_SEED = 42
TEST_FRAC = 0.2

def try_get_items(ds):
    """
    Returns (paths, labels) if the dataset exposes lightweight metadata.
    Otherwise returns (None, None).
    """
    if hasattr(ds, "items"):
        try:
            paths = [p for p, y in ds.items]
            labels = [int(y) for p, y in ds.items]
            return paths, labels
        except Exception:
            pass
    # Some versions might use .samples or .index — add probes here if needed
    return None, None

def get_labels_and_optional_paths(ds):
    paths, labels = try_get_items(ds)
    if labels is not None:
        return paths, labels

    # Fallback (slower): index the dataset (could load arrays). We’ll stop if it’s too slow for you.
    print("Falling back to indexing to read labels (may take a while)…")
    labels = []
    maybe_paths = []
    for i in range(len(ds)):
        x, y = ds[i]
        try:
            y_val = int(getattr(y, "item", lambda: y)())
        except Exception:
            y_val = int(y)
        labels.append(y_val)
        # If the item returns a path somewhere (some datasets do), keep it; otherwise None
        maybe_paths.append(None)
    return maybe_paths, labels

paths, labels = get_labels_and_optional_paths(dataset)
num_classes = len(set(labels))
print(f"Found labels for {len(labels)} items across {num_classes} classes.")

counts = Counter(labels)

for cls in sorted(CLASS_LABELS.keys()):
    n = counts.get(cls, 0)
    pct = (n / total * 100) if total else 0.0
    print(f"Class {cls:>1} – {CLASS_LABELS[cls]:<22}: {n:>6}  ({pct:6.2f}%)")

# Also report anything outside your known mapping (just in case)
unknown_classes = sorted(k for k in counts.keys() if k not in CLASS_LABELS)
if unknown_classes:
    print("\n⚠️ Found labels not in CLASS_LABELS:")
    for cls in unknown_classes:
        n = counts[cls]
        pct = (n / total * 100) if total else 0.0
        print(f"Class {cls}: {n} ({pct:.2f}%)")